# Hard-label audit and clean-hard benchmark

This notebook does not train or calibrate any model. It audits the fixed
hard labels, builds mutually exclusive `hard_clean`, `hard_suspicious`,
and `hard_conflicting` subsets, and evaluates frozen S0, S2, and
S2+CatBoost predictions.

Subset selection is independent of model predictions. Scores are attached
only after the label/representation flags have been computed.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "scripts/audit_hard_clean.py").is_file():
    ROOT = ROOT.parent
OUTPUT = ROOT / "reports/minilm_s2_hard_clean_audit"
assert (ROOT / "scripts/audit_hard_clean.py").is_file(), ROOT

## Run the deterministic audit

In [2]:
completed = subprocess.run(
    [sys.executable, str(ROOT / "scripts/audit_hard_clean.py")],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
if completed.returncode:
    print(completed.stdout)
    print(completed.stderr, file=sys.stderr)
    completed.check_returncode()
print("Audit completed:", OUTPUT)

Audit completed: C:\Users\Professional\product_matching\reports\minilm_s2_hard_clean_audit


## Objective contradictions and benchmark subsets

`definite_label_conflict` means opposite targets for the same unordered
ID pair or for the same unordered pair of normalized full item
representations. Suspicious flags are not automatic label corrections.

In [3]:
audit_summary = pd.read_csv(OUTPUT / "label_audit_summary.csv")
display(audit_summary)

,subset,pairs,positives,prevalence,categories,definite_label_conflicts,unordered_id_pair_target_conflicts,representation_pair_target_conflicts,suspicious_negative_identity,suspicious_positive_conflict,sku_vs_human_title
0,hard_all,5814,1481,0.254730,18,3,0,3,204,679,622
1,hard_clean,4929,802,0.162710,18,0,0,0,0,0,494
2,hard_suspicious,882,678,0.768707,18,0,0,0,204,678,128
3,hard_conflicting,3,1,0.333333,1,3,0,3,0,1,0


## Frozen-model PR-AUC

In [4]:
metrics = pd.read_csv(OUTPUT / "benchmark_metrics.csv")
display(
    metrics.pivot(
        index=["subset", "pairs", "prevalence"],
        columns="model",
        values="macro_average_precision",
    ).reset_index()
)
display(
    metrics.pivot(
        index=["subset", "pairs", "prevalence"],
        columns="model",
        values="macro_roc_auc",
    ).reset_index()
)

model,subset,pairs,prevalence,S0,S2,S2_CATBOOST
0,hard_all,5814,0.254730,0.307657,0.307319,0.314517
1,hard_clean,4929,0.162710,0.221657,0.236681,0.247444
2,hard_conflicting,3,0.333333,0.333333,0.333333,0.333333
3,hard_suspicious,882,0.768707,0.886120,0.831469,0.855665
4,iid,12000,0.259833,0.683217,0.738074,0.736722
5,ood,41171,0.222365,0.503071,0.598185,0.600112


model,subset,pairs,prevalence,S0,S2,S2_CATBOOST
0,hard_all,5814,0.254730,0.594694,0.584716,0.584495
1,hard_clean,4929,0.162710,0.604754,0.639052,0.643890
2,hard_conflicting,3,0.333333,0.500000,0.500000,0.500000
3,hard_suspicious,882,0.768707,0.594822,0.534667,0.632507
4,iid,12000,0.259833,0.854433,0.881558,0.882735
5,ood,41171,0.222365,0.771433,0.831777,0.835573


Absolute PR-AUC values must not be compared naively across subsets with
different prevalence. The primary model comparison is within each row.
ROC-AUC is shown as a prevalence-insensitive ranking diagnostic.

## Clean-hard diagnostic slices

In [5]:
clean_slices = pd.read_csv(OUTPUT / "hard_clean_slice_metrics.csv")
display(clean_slices)

,subset,model,pairs,positives,prevalence,categories,eligible_categories,macro_average_precision,macro_roc_auc,overall_average_precision,overall_roc_auc,positive_mean_score,negative_mean_score,fnr_at_0_5,fpr_at_0_5,slice
0,hard_clean::numeric_conflict,S0,932,0,0.000000,18,0,NaN,NaN,NaN,NaN,NaN,0.110094,NaN,0.080472,numeric_conflict
1,hard_clean::numeric_conflict,S2,932,0,0.000000,18,0,NaN,NaN,NaN,NaN,NaN,0.109232,NaN,0.083691,numeric_conflict
2,hard_clean::numeric_conflict,S2_CATBOOST,932,0,0.000000,18,0,NaN,NaN,NaN,NaN,NaN,0.101859,NaN,0.076180,numeric_conflict
3,hard_clean::code_conflict,S0,428,0,0.000000,16,0,NaN,NaN,NaN,NaN,NaN,0.081377,NaN,0.044393,code_conflict
4,hard_clean::code_conflict,S2,428,0,0.000000,16,0,NaN,NaN,NaN,NaN,NaN,0.099475,NaN,0.079439,code_conflict
5,hard_clean::code_conflict,S2_CATBOOST,428,0,0.000000,16,0,NaN,NaN,NaN,NaN,NaN,0.093217,NaN,0.065421,code_conflict
6,hard_clean::model_conflict,S0,380,0,0.000000,13,0,NaN,NaN,NaN,NaN,NaN,0.222365,NaN,0.139474,model_conflict
7,hard_clean::model_conflict,S2,380,0,0.000000,13,0,NaN,NaN,NaN,NaN,NaN,0.256090,NaN,0.210526,model_conflict
8,hard_clean::model_conflict,S2_CATBOOST,380,0,0.000000,13,0,NaN,NaN,NaN,NaN,NaN,0.250347,NaN,0.213158,model_conflict
9,hard_clean::sku_vs_human_title,S0,494,139,0.281377,18,17,0.397133,0.430893,0.351310,0.554970,0.368031,0.311728,0.705036,0.228169,sku_vs_human_title


## Up to 100 examples per audit issue

In [6]:
contradictions = pd.read_csv(OUTPUT / "label_contradictions.csv")
display(contradictions.groupby("audit_issue").size().rename("examples"))
display(contradictions.head(100))

audit_issue
definite_label_conflict                    3
negative_exact_normalized_title          100
positive_code_conflict                   100
positive_critical_attribute_conflict     100
positive_model_code_conflict              95
positive_numeric_conflict                100
sku_vs_human_title                       100
very_high_lexical_similarity_negative    100
very_low_lexical_similarity_positive     100
Name: examples, dtype: int64

,audit_issue,id1,id2,canonical_id1,canonical_id2,target,category,hard_subset,title1,title2,...,title_char_tfidf_cosine,numeric_context_conflict_count,unit_conflict_count,brand_conflict,model_conflict,memory_conflict,color_conflict,s0_score,s2_score,catboost_score
0,definite_label_conflict,223338314794,377957183557,223338314794,377957183557,1.0,Детские товары,hard_conflicting,мягкая плюшевая игрушка подушка гусь обнимусь ...,asras - toy / мягкая игрушка подушка гусь обни...,...,0.691417,0,0,0.0,0.0,0.0,1.0,0.823151,0.212306,0.170806
1,definite_label_conflict,377957183557,240518299856,240518299856,377957183557,0.0,Детские товары,hard_conflicting,asras - toy / мягкая игрушка подушка гусь обни...,мягкая плюшевая игрушка подушка гусь обнимусь ...,...,0.691417,0,0,0.0,0.0,0.0,1.0,0.823151,0.212306,0.170806
2,definite_label_conflict,730144555490,377957183557,377957183557,730144555490,0.0,Детские товары,hard_conflicting,мягкая плюшевая игрушка подушка гусь обнимусь ...,asras - toy / мягкая игрушка подушка гусь обни...,...,0.691417,0,0,0.0,0.0,0.0,1.0,0.823151,0.212306,0.170806
3,negative_exact_normalized_title,34359876498,111669167130,34359876498,111669167130,0.0,Дом и сад,hard_suspicious,трусы,трусы,...,1.000000,0,0,0.0,1.0,0.0,0.0,0.041852,0.015968,0.006479
4,negative_exact_normalized_title,283467899562,197568544190,197568544190,283467899562,0.0,Обувь,hard_suspicious,кроссовки nike jordan,кроссовки nike jordan,...,1.000000,0,0,0.0,1.0,0.0,1.0,0.028653,0.054077,0.016713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,negative_exact_normalized_title,721554629218,498216323484,498216323484,721554629218,0.0,Канцелярские товары,hard_suspicious,скетчбук,скетчбук,...,1.000000,0,0,0.0,0.0,0.0,1.0,0.036357,0.005829,0.006874
96,negative_exact_normalized_title,678604848226,506806242554,506806242554,678604848226,0.0,Мебель,hard_suspicious,тумба под телевизор,тумба под телевизор,...,1.000000,0,0,0.0,0.0,0.0,1.0,0.018229,0.001154,0.004689
97,negative_exact_normalized_title,807453990178,506806281731,506806281731,807453990178,0.0,Обувь,hard_suspicious,кеды makfly,кеды makfly,...,1.000000,0,0,0.0,0.0,0.0,1.0,0.101056,0.007422,0.008550
98,negative_exact_normalized_title,523986125362,833223663864,523986125362,833223663864,0.0,Обувь,hard_suspicious,сланцы roxy,сланцы roxy,...,1.000000,0,0,0.0,0.0,0.0,1.0,0.057599,0.002407,0.007341


## Interpretation

A higher clean-hard ROC-AUC than hard-all indicates that ambiguous labels
materially damage ranking evaluation. If clean-hard remains far below IID,
the hard-set failure is not explained by label noise alone: selection and
genuinely difficult candidate pairs remain important.